# Tutorial 10: Segmentation using U-Net (Custom Model)

## Objectives
- Understand the architecture of U-Net
- Implementing U-Net from Scratch
- Visualizing the results

This notebook is ready for **Google Colab** (Runtime → Change runtime type → **GPU** recommended).

We first follow the tutorial with synthetic data, then complete the **Tasks** using real labelled data from Kaggle (and a convenient TFDS alternative).

## 1. Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from glob import glob

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 2. Prepare the Dataset (Synthetic – Tutorial Example)

We create a simple synthetic dataset of white rectangles on black background for quick experimentation.

In [ ]:
def create_sample_data(img_size=128, num_samples=100):
    images = []
    masks = []
    for _ in range(num_samples):
        img = np.zeros((img_size, img_size, 3), dtype=np.uint8)
        mask = np.zeros((img_size, img_size, 1), dtype=np.uint8)

        # Add a random rectangle
        x1, y1 = np.random.randint(0, img_size // 2, 2)
        x2 = x1 + np.random.randint(img_size // 4, img_size // 2)
        y2 = y1 + np.random.randint(img_size // 4, img_size // 2)
        x2 = min(x2, img_size - 1)
        y2 = min(y2, img_size - 1)

        img = cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 255), -1)
        mask = cv2.rectangle(mask, (x1, y1), (x2, y2), 1, -1)

        images.append(img)
        masks.append(mask)

    return np.array(images) / 255.0, np.array(masks).astype(np.float32)

# Generate synthetic data
images, masks = create_sample_data(img_size=128, num_samples=200)
print(f"Images shape: {images.shape}, Masks shape: {masks.shape}")

# Quick visualization
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(images[0])
plt.title("Sample Image")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(masks[0].squeeze(), cmap="gray")
plt.title("Sample Mask")
plt.axis("off")
plt.show()

## 3. Define the U-Net Model (from Scratch)

In [ ]:
def unet(input_size=(128, 128, 3), num_filters=64):
    inputs = layers.Input(input_size)

    # Encoder
    c1 = layers.Conv2D(num_filters, (3, 3), activation="relu", padding="same")(inputs)
    c1 = layers.Conv2D(num_filters, (3, 3), activation="relu", padding="same")(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(num_filters * 2, (3, 3), activation="relu", padding="same")(p1)
    c2 = layers.Conv2D(num_filters * 2, (3, 3), activation="relu", padding="same")(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    # Bottleneck
    c3 = layers.Conv2D(num_filters * 4, (3, 3), activation="relu", padding="same")(p2)
    c3 = layers.Conv2D(num_filters * 4, (3, 3), activation="relu", padding="same")(c3)

    # Decoder
    u1 = layers.UpSampling2D((2, 2))(c3)
    u1 = layers.concatenate([u1, c2])
    c4 = layers.Conv2D(num_filters * 2, (3, 3), activation="relu", padding="same")(u1)
    c4 = layers.Conv2D(num_filters * 2, (3, 3), activation="relu", padding="same")(c4)

    u2 = layers.UpSampling2D((2, 2))(c4)
    u2 = layers.concatenate([u2, c1])
    c5 = layers.Conv2D(num_filters, (3, 3), activation="relu", padding="same")(u2)
    c5 = layers.Conv2D(num_filters, (3, 3), activation="relu", padding="same")(c5)

    outputs = layers.Conv2D(1, (1, 1), activation="sigmoid")(c5)

    return Model(inputs, outputs, name="U-Net")

# Build the model
model = unet()
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 4. Train the Model (Synthetic Data)

In [ ]:
# Split data into training and validation sets
train_images, val_images = images[:160], images[160:]
train_masks, val_masks = masks[:160], masks[160:]

# Train the U-Net model
history = model.fit(
    train_images, train_masks,
    validation_data=(val_images, val_masks),
    batch_size=8,
    epochs=15,
    verbose=1
)

## 5. Evaluate and Visualize Results

In [ ]:
# Evaluate the model on validation data
loss, accuracy = model.evaluate(val_images, val_masks, verbose=0)
print(f"Validation Loss: {loss:.4f}, Validation Accuracy: {accuracy:.4f}")

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Loss")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Train Acc")
plt.plot(history.history["val_accuracy"], label="Val Acc")
plt.title("Accuracy")
plt.legend()
plt.show()

In [ ]:
# Predict on a few validation samples
pred_masks = model.predict(val_images, verbose=0)

def plot_sample(image, true_mask, pred_mask, threshold=0.5):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.title("Input Image")
    plt.imshow(image)
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.title("True Mask")
    plt.imshow(true_mask.squeeze(), cmap="gray")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.title("Predicted Mask")
    plt.imshow((pred_mask.squeeze() > threshold).astype(np.uint8), cmap="gray")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# Show a few sample results
for i in range(3):
    plot_sample(val_images[i], val_masks[i], pred_masks[i])

---
# Tasks

## Task Overview
1. Use real labelled data from **Kaggle**
2. Train / test the U-Net on that data (optionally modify layers)
3. Experiment with different learning rates and visualize results

We will use the classic **Carvana Image Masking Challenge** dataset (mentioned in the tutorial) via the Kaggle API, plus a convenient alternative (Oxford-IIIT Pet) that is also available on Kaggle / TFDS.

## Task 1 & 2: Load labelled data from Kaggle

### Setup Kaggle API in Colab

1. Go to https://www.kaggle.com/ → Account → Create New API Token (downloads `kaggle.json`)
2. Upload `kaggle.json` in the cell below
3. Accept the competition rules for Carvana if prompted (https://www.kaggle.com/c/carvana-image-masking-challenge)

In [ ]:
# Install and configure Kaggle
!pip install -q kaggle

from google.colab import files
import zipfile

# Upload your kaggle.json
print("Please upload your kaggle.json file")
uploaded = files.upload()  # select kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API configured!")

### Download Carvana dataset (or a smaller alternative)

Carvana is large (~ few GB). For a quick demo we download only a subset or use a lighter Kaggle dataset.

**Option A – Carvana (full challenge data)**  
```python
!kaggle competitions download -c carvana-image-masking-challenge -p ./carvana --force
```

**Option B – Recommended for Colab (smaller, still labelled segmentation)**  
We use a convenient packaged dataset or the Oxford Pets (also on Kaggle). Below we use **TensorFlow Datasets** for Oxford-IIIT Pet (has high-quality segmentation masks) which is very easy and free of competition rules. You can also search “oxford iiit pet” or “carvana” on Kaggle and download similarly.

In [ ]:
# ===== Practical labelled data: Oxford-IIIT Pet (via TFDS) =====
# This dataset has pixel-level segmentation masks and is widely used.
# It is also available on Kaggle under various uploads.

!pip install -q tensorflow-datasets

import tensorflow_datasets as tfds

dataset, info = tfds.load(
    "oxford_iiit_pet:3.*.*",
    with_info=True,
    download=True
)
print(info)

# The masks contain 3 classes: 1=pet, 2=border, 3=background
# We will convert to binary (pet vs background) for simplicity with our U-Net.

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 16

def normalize_and_resize(sample):
    image = tf.image.resize(sample["image"], (IMG_SIZE, IMG_SIZE))
    mask = tf.image.resize(sample["segmentation_mask"], (IMG_SIZE, IMG_SIZE))

    image = tf.cast(image, tf.float32) / 255.0
    # Convert to binary: pet (class 1) = 1, everything else = 0
    mask = tf.cast(mask == 1, tf.float32)
    return image, mask

train_ds = (
    dataset["train"]
    .map(normalize_and_resize, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    dataset["test"]
    .map(normalize_and_resize, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Peek at a batch
for images_batch, masks_batch in train_ds.take(1):
    print("Batch images:", images_batch.shape)
    print("Batch masks:", masks_batch.shape)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(images_batch[0])
    plt.title("Pet Image")
    plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.imshow(masks_batch[0].numpy().squeeze(), cmap="gray")
    plt.title("Segmentation Mask (binary)")
    plt.axis("off")
    plt.show()
    break

### (Optional) Download Carvana via Kaggle
Uncomment and run after accepting competition rules if you want the original Carvana data.

In [ ]:
# !kaggle competitions download -c carvana-image-masking-challenge -p ./carvana
# !unzip -q ./carvana/*.zip -d ./carvana
# Then write a custom tf.data pipeline reading the .jpg images and .gif masks.

## Task 2 continued: Train U-Net on real data + optionally change layers

In [ ]:
# Rebuild a slightly deeper / modified U-Net (extra filters + dropout for regularization)
def unet_modified(input_size=(128, 128, 3)):
    inputs = layers.Input(input_size)

    # Encoder
    c1 = layers.Conv2D(64, 3, activation="relu", padding="same")(inputs)
    c1 = layers.Conv2D(64, 3, activation="relu", padding="same")(c1)
    c1 = layers.Dropout(0.1)(c1)
    p1 = layers.MaxPooling2D(2)(c1)

    c2 = layers.Conv2D(128, 3, activation="relu", padding="same")(p1)
    c2 = layers.Conv2D(128, 3, activation="relu", padding="same")(c2)
    c2 = layers.Dropout(0.1)(c2)
    p2 = layers.MaxPooling2D(2)(c2)

    c3 = layers.Conv2D(256, 3, activation="relu", padding="same")(p2)
    c3 = layers.Conv2D(256, 3, activation="relu", padding="same")(c3)
    p3 = layers.MaxPooling2D(2)(c3)

    # Bottleneck
    b = layers.Conv2D(512, 3, activation="relu", padding="same")(p3)
    b = layers.Conv2D(512, 3, activation="relu", padding="same")(b)

    # Decoder
    u3 = layers.UpSampling2D(2)(b)
    u3 = layers.concatenate([u3, c3])
    c4 = layers.Conv2D(256, 3, activation="relu", padding="same")(u3)
    c4 = layers.Conv2D(256, 3, activation="relu", padding="same")(c4)

    u2 = layers.UpSampling2D(2)(c4)
    u2 = layers.concatenate([u2, c2])
    c5 = layers.Conv2D(128, 3, activation="relu", padding="same")(u2)
    c5 = layers.Conv2D(128, 3, activation="relu", padding="same")(c5)

    u1 = layers.UpSampling2D(2)(c5)
    u1 = layers.concatenate([u1, c1])
    c6 = layers.Conv2D(64, 3, activation="relu", padding="same")(u1)
    c6 = layers.Conv2D(64, 3, activation="relu", padding="same")(c6)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(c6)
    return Model(inputs, outputs, name="U-Net-Modified")

model_real = unet_modified()
model_real.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5)]
)
model_real.summary()

In [ ]:
# Train on real data (a few epochs for demo – increase for better results)
history_real = model_real.fit(
    train_ds,
    validation_data=test_ds,
    epochs=8,
    verbose=1
)

In [ ]:
# Visualize predictions on real data
def show_predictions(model, dataset, num=3):
    for images, masks in dataset.take(1):
        preds = model.predict(images, verbose=0)
        for i in range(min(num, images.shape[0])):
            plt.figure(figsize=(12, 4))
            plt.subplot(1, 3, 1)
            plt.imshow(images[i])
            plt.title("Input")
            plt.axis("off")
            plt.subplot(1, 3, 2)
            plt.imshow(masks[i].numpy().squeeze(), cmap="gray")
            plt.title("True Mask")
            plt.axis("off")
            plt.subplot(1, 3, 3)
            plt.imshow((preds[i].squeeze() > 0.5).astype(np.uint8), cmap="gray")
            plt.title("Predicted")
            plt.axis("off")
            plt.tight_layout()
            plt.show()
        break

show_predictions(model_real, test_ds)

## Task 3: Different learning rates and visualize results

In [ ]:
learning_rates = [1e-3, 1e-4, 1e-5]
histories = {}

for lr in learning_rates:
    print(f"\n===== Training with learning rate = {lr} =====")
    m = unet(input_size=(128, 128, 3))  # original simpler U-Net for fair comparison
    m.compile(
        optimizer=Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    hist = m.fit(
        train_ds,
        validation_data=test_ds,
        epochs=5,          # keep short for demo; increase for real experiments
        verbose=1
    )
    histories[lr] = hist.history

In [ ]:
# Visualize effect of different learning rates
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
for lr, h in histories.items():
    plt.plot(h["val_loss"], label=f"lr={lr}")
plt.title("Validation Loss vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
for lr, h in histories.items():
    plt.plot(h["val_accuracy"], label=f"lr={lr}")
plt.title("Validation Accuracy vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print("Observation tips:")
print("- Too high LR (1e-3) often oscillates or diverges")
print("- 1e-4 is usually a solid default for U-Net")
print("- Very low LR (1e-5) converges slowly")

---
## Summary

| Part              | Content                                      |
|-------------------|----------------------------------------------|
| Tutorial          | Synthetic data + classic U-Net from scratch  |
| Task 1            | Real labelled data (Kaggle / Oxford Pets)    |
| Task 2            | Train on real data + modified deeper U-Net   |
| Task 3            | Compare multiple learning rates              |

**Next ideas**
- Add Dice loss or combined BCE + Dice for better segmentation metrics
- Try data augmentation (`tf.image` random flip/rotate)
- Increase image size / model capacity for higher quality masks
- Download the full Carvana dataset once you have more GPU time / disk